In [ ]:
# @title Install
!pip install -q "diffusers>=0.33" transformers accelerate imageio imageio-ffmpeg decord

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline
from PIL import Image
import imageio.v2 as imageio
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| cuda:', torch.cuda.is_available())

In [ ]:
# @title Load model (LTX-Video 2B Distilled, FREE, image-to-video)
MODEL_ID = 'Lightricks/LTX-Video-2B-0.9.6-Distilled-04-25'
pipe = LTXImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).to(device)
print('model loaded:', MODEL_ID)

In [ ]:
# @title Inputs (upload vault_door.jpeg to the left sidebar first)
IMAGE_PATH = '/content/vault_door.jpeg'  # change if named differently
assert os.path.exists(IMAGE_PATH), 'Upload vault_door.jpeg first (left sidebar -> upload)'
img = Image.open(IMAGE_PATH).convert('RGB')
# Match REFERENCE aspect: reference is 720x1280 (vertical paper card). Use 704x1280 (32-divisible for LTX).
W, H = 704, 1280
img = img.resize((W, H))
print('input size', img.size)

In [ ]:
# @title Prompt (steer toward paper-animation / Vox-explainer look)
prompt = (
    'Paper-collage animation style, flat illustrated vault door made of cut paper layers, '
    'the paper pieces shift and lift slightly with a gentle hand-made stop-motion drift, '
    'soft paper shadows, warm editorial color palette, Vox-style explainer motion, '
    'subtle slow camera push-in, no 3D, no photorealism'
)
negative_prompt = 'photorealistic, 3d render, CGI, jitter, flicker, distorted, blurry'

In [ ]:
# @title Generate (~5s at 24fps)
result = pipe(
    image=img,
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=H, width=W,
    num_frames=121,
    num_inference_steps=30,
    generator=torch.Generator(device=device).manual_seed(42),
)
frames = result.frames[0]

In [ ]:
# @title Save + download
OUT = '/content/ltx_vault.mp4'
writer = imageio.get_writer(OUT, fps=24, codec='libx264', macro_block_size=None)
for f in frames:
    writer.append_data(f)
writer.close()
print('saved', OUT, 'frames:', len(frames))
from google.colab import files
files.download(OUT)
print('Download triggered. Move to ~/Downloads/vox/ltx_vault.mp4 and ask Hermes to verify >=60%.')